# Milestone 1: Exploratory Data Analysis
Heart Disease Risk Prediction (BRFSS 2022)

Covers the September milestone:
- feature typing / encoding decisions
- missing data audit
- correlation matrix + target relationships
- distributions, bivariate analysis, summary stats
- fixed-seed, stratified train/validation/test split


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

RANDOM_SEED = 42  # fixed seed used everywhere below for reproducibility

In [ ]:
DATA_PATH = "../Data/heart_2022_with_nans.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(445132, 40)


,State,Sex,GeneralHealth,PhysicalHealthDays,MentalHealthDays,LastCheckupTime,PhysicalActivities,SleepHours,RemovedTeeth,HadHeartAttack,HadAngina,HadStroke,HadAsthma,HadSkinCancer,HadCOPD,HadDepressiveDisorder,HadKidneyDisease,HadArthritis,HadDiabetes,DeafOrHardOfHearing,BlindOrVisionDifficulty,DifficultyConcentrating,DifficultyWalking,DifficultyDressingBathing,DifficultyErrands,SmokerStatus,ECigaretteUsage,ChestScan,RaceEthnicityCategory,AgeCategory,HeightInMeters,WeightInKilograms,BMI,AlcoholDrinkers,HIVTesting,FluVaxLast12,PneumoVaxEver,TetanusLast10Tdap,HighRiskLastYear,CovidPos
0,Alabama,Female,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,No,8.0,NaN,No,No,No,No,No,No,No,No,No,Yes,No,No,No,No,No,No,Never smoked,Not at all (right now),No,"White only, Non-Hispanic",Age 80 or older,NaN,NaN,NaN,No,No,Yes,No,"Yes, received tetanus shot but not sure what type",No,No
1,Alabama,Female,Excellent,0.0,0.0,NaN,No,6.0,NaN,No,No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,Never smoked,Never used e-cigarettes in my entire life,No,"White only, Non-Hispanic",Age 80 or older,1.60,68.04,26.57,No,No,No,No,"No, did not receive any tetanus shot in the pa...",No,No
2,Alabama,Female,Very good,2.0,3.0,Within past year (anytime less than 12 months ...,Yes,5.0,NaN,No,No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,Never smoked,Never used e-cigarettes in my entire life,No,"White only, Non-Hispanic",Age 55 to 59,1.57,63.50,25.61,No,No,No,No,NaN,No,Yes
3,Alabama,Female,Excellent,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,7.0,NaN,No,No,No,Yes,No,No,No,No,Yes,No,No,No,No,No,No,No,Current smoker - now smokes some days,Never used e-cigarettes in my entire life,Yes,"White only, Non-Hispanic",NaN,1.65,63.50,23.30,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No
4,Alabama,Female,Fair,2.0,0.0,Within past year (anytime less than 12 months ...,Yes,9.0,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Never smoked,Never used e-cigarettes in my entire life,Yes,"White only, Non-Hispanic",Age 40 to 44,1.57,53.98,21.77,Yes,No,No,Yes,"No, did not receive any tetanus shot in the pa...",No,No


## 2. Feature typing
Document each column's storage dtype and its semantic type (numerical continuous,
numerical discrete, binary categorical, multi-class categorical, free text).
This will help determine how to encode it later.

In [27]:
dtype_summary = pd.DataFrame({
    'dtype': df.dtypes,
    'n_unique': df.nunique(),
    'sample_values': [df[c].dropna().unique()[:5] for c in df.columns]
})
dtype_summary

,dtype,n_unique,sample_values
State,object,54,"[Alabama, Alaska, Arizona, Arkansas, California]"
Sex,object,2,"[Female, Male]"
GeneralHealth,object,5,"[Very good, Excellent, Fair, Poor, Good]"
PhysicalHealthDays,float64,31,"[0.0, 2.0, 1.0, 8.0, 5.0]"
MentalHealthDays,float64,31,"[0.0, 3.0, 9.0, 5.0, 15.0]"
LastCheckupTime,object,4,[Within past year (anytime less than 12 months...
PhysicalActivities,object,2,"[No, Yes]"
SleepHours,float64,24,"[8.0, 6.0, 5.0, 7.0, 9.0]"
RemovedTeeth,object,4,"[None of them, 1 to 5, 6 or more, but not all,..."
HadHeartAttack,object,2,"[No, Yes]"


In [28]:
# TODO: after reviewing dtype_summary above, classify each column

## 3. Missing data audit
For each column: how many values are missing, and what's a defensible handling strategy
(drop rows, impute mean/median/mode, impute with a model, or add a "missing" indicator
category)?

In [31]:
missing = df.isna().sum().sort_values(ascending=False)
missing_summary = pd.DataFrame({'n_missing': missing})
missing_summary[missing_summary['n_missing'] > 0]

,n_missing
TetanusLast10Tdap,82516
PneumoVaxEver,77040
HIVTesting,66127
ChestScan,56046
CovidPos,50764
HighRiskLastYear,50623
BMI,48806
FluVaxLast12,47121
AlcoholDrinkers,46574
WeightInKilograms,42078


In [32]:
# TODO: document your per-column decision

## 4. Correlation matrix
Numeric-only Pearson correlation first, then rank features by their correlation with the
target. For categorical predictors, a full correlation matrix isn't meaningful as-is --
consider encoding them before including them.

In [33]:
# Encode target, then rank features by correlation with it
target_col = 'HadHeartAttack'

## 5. Distributions, bivariate analysis, and summary statistics

In [34]:
df[numeric_cols].describe().T  # mean, std, min, max, quartiles for every numeric column

,count,mean,std,min,25%,50%,75%,max
PhysicalHealthDays,434205.0,4.347919,8.688912,0.00,0.00,0.00,3.00,30.00
MentalHealthDays,436065.0,4.382649,8.387475,0.00,0.00,0.00,5.00,30.00
SleepHours,439679.0,7.022983,1.502425,1.00,6.00,7.00,8.00,24.00
HeightInMeters,416480.0,1.702691,0.107177,0.91,1.63,1.70,1.78,2.41
WeightInKilograms,403054.0,83.074470,21.448173,22.68,68.04,80.74,95.25,292.57
BMI,396326.0,28.529842,6.554889,12.02,24.13,27.44,31.75,99.64


In [35]:
# Univariate: distribution of each numeric feature

In [36]:
# Bivariate: each numeric feature vs. target

In [37]:
# Bivariate for categorical features

## 6. Train / validation / test split

In [38]:
# First split off the test set, then split train/val from what remains